In [1]:
import pdfplumber
import os
from itertools import islice
from pathlib import Path
from typing import Dict, List

In [2]:
import json

In [3]:
BASE_FOLDER = "lic_policies"

In [4]:
from pathlib import Path

In [5]:
# List down the sub-directories under base folder
# List down the files for each directory
# Each sub-directory is category
# Each file name is plan
# Read each file and put it under text

In [6]:
pdfile = Path("lic_policies/endowment_plans/bima_lakshmi.pdf")

In [7]:
# with pdfplumber.open(pdfile) as pdf:
#     for pageno, page in enumerate(pdf.pages):
#         text = page.extract_text()
#         if pageno + 1 == 2:
#             text = text[:text.rindex('\n')]
#             print(text)
#         else:
#             continue

In [8]:
# def is_table_empty(tables):
#     #print(tables)
#     for table in tables:
#         for row, data in enumerate(table[1:]):
#             print(data)
#     return True

In [9]:
UNICODE_MAP = {
    "\u2013": "-", # en dash
    "\u2014": "-", # em dash
    "\u201c": '"', # left double quote
    "\u201d": '"', # right double quote
    "\u2018": "'", # left single quote
    "\u2019": "'"  # right single quote
}

In [10]:
def is_covered_by_object(page, threshold=0.85):
    page_area = page.width * page.height
    object_area = 0
    tables = page.find_tables()
    
    for table in tables:
        x0, top, x1, bottom = table.bbox
        object_area += (x1 - x0) +(bottom - top)
    
    for rect in page.rects:
        object_area += (rect['x1'] - rect['x0']) * (rect['y1'] - rect['y0']) * (rect['bottom'] - rect['top'])
        
    coverage_ratio = object_area / page_area
    return coverage_ratio >= threshold

In [11]:
# with pdfplumber.open(pdfile) as pdf:
#     for pageno, page in enumerate(pdf.pages):
#         tables = page.extract_tables()
#         if pageno + 1 == 4:
#             text = page.extract_text()
#             #text = text[:text.rindex('\n')]#.replace("Thank you", "").replace("Thanking you.", "").replace("_", "")
#             text = text[:text.rindex('\n')]
#             #print(page.rects[2])
# #             tbls = page.find_tables()
# # #             for tb in tbls:
# #                 print(tb.bbox)
#             #print(tbls.bbox)
#             print(is_covered_by_object(page))
#             for line in text.split('\n'):
#                 #print(line)
#                 if line.strip()[-1] == ':' or line.strip() == '':
#                     text = text.replace(f"{line}\n", '')
#             # print(page.images)
#             # print(page.extract_tables())
#             print("\n\n")
#             print(text)
# #             print(len(tables))
# #             print(page.images)
# #             print(tables)
# #             table1 = [is_table_empty(table) for table in tables]
# #             non_empty = [table for table in tables if is_table_empty(table)]
# #             print(len(non_empty))
#         else:
#             continue
# #         text = page.extract_text()
# #         if pageno + 1 == 4:
# #             text = text[:text.rindex('\n')]
# #             print(text)
# #         else:
# #             continue

In [12]:
##CLEANING STEPS

# if table/rect coverage > threshold(85%) then page_type = 'Form'
# remove footer of the page
# remove empty fields i.e., where last char of the line.strip() is ':'

In [13]:
def extract_text_from_file(filepath: str) -> List[dict]:
    text_by_page = []
    with pdfplumber.open(filepath) as pdf:
        for pageno, page in enumerate(pdf.pages):            
            original_text = page.extract_text()
            if original_text and original_text.strip():
                if original_text.rfind('\n') == -1 and len(page.extract_text_lines()) == 1:
                    cleaned_text = original_text.replace(page.extract_text_lines()[-1]['text'], '')
                else:
                    cleaned_text = original_text[:original_text.rindex('\n')]
                    for line in cleaned_text.split('\n'):
                        if line.strip()[-1] == ':' or line.strip() == '':
                            cleaned_text = cleaned_text.replace(f"{line}\n", '')
                            
                if cleaned_text == "":
                    continue
                    
                for char, replacement in UNICODE_MAP.items():
                    cleaned_text = cleaned_text.replace(char, replacement)
                
                page_type = 'form' if is_covered_by_object(page) else 'content'
                if page_type == 'content' and ('Annexure' in cleaned_text[:50] or 'Schedule' in cleaned_text[:50]):
                    page_type = 'annexure'
                elif page_type == 'content' and len(cleaned_text) <=50:
                    page_type = 'low_content'
                    
#                 print(filepath, pageno + 1)
#                 for line in text.split('\n'):
#                     if line.strip()[-1] == ':' or line.strip() == '':
#                         text = text.replace(f"{line}\n", '')
                text_by_page.append({
                    "page": pageno + 1,
                    "page_type": page_type,
                    "cleaned_text": cleaned_text.strip(),
                    "original_text": original_text
                })
    return text_by_page

In [14]:
def create_json(documents: List):
    with open("policies.json", "w") as file:
        json.dump(documents, file, indent=2)

In [15]:
def standardize_policy_files(location: str) -> List[dict]:
    all_docs = []
    dirs = os.listdir(location)
    for category in dirs:
        path = Path(f"{location}/{category}")
        files = [file for file in path.iterdir() if file.is_file()]
        for file in files:
            if file.suffix != ".pdf":
                continue
            file_content = extract_text_from_file(file)
            all_docs.append({
                "policy_name": file.name.replace(".pdf", ""),
                "category": category,
                "source_path": str(path),
                "pages": file_content
            })
    return all_docs

In [16]:
documents = standardize_policy_files(BASE_FOLDER)

In [17]:
#documents

In [18]:
create_json(documents)

In [19]:
# with pdfplumber.open(pdfile) as pdf:
#     for pgno, page in enumerate(pdf.pages):
#         if pgno + 1 == 20:
#             text = page.extract_text()
#             for line in text.split('\n'):
#                 if line.strip()[-1] == ':' or line.strip() == '':
#                     text = text.replace(f"{line}\n", '')

In [20]:
# s = 'LIC\u2013 is the insurance year'
# s.replace()